## Customer Segmentation 

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [2]:
data = pd.read_excel("Online Retail.xlsx")

In [3]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
print(data.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [5]:
data = data.dropna(subset=["CustomerID"])

In [6]:
print(data.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [7]:
duplicate_count = data.duplicated().sum()

In [8]:
print(duplicate_count)

5225


In [9]:
data = data.drop_duplicates()

In [10]:
duplicate_check = data.duplicated().sum()

In [11]:
print(duplicate_check)

0


In [12]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

In [13]:
data["Spending"] = data["Quantity"] * data["UnitPrice"]

In [14]:
customer_data = data.groupby("CustomerID").agg(  
                Total_Quantity = ("Quantity" , "sum"),
                Total_Spending = ("Spending" , "sum" ),
                Purchase_Count = ("InvoiceDate" , "count"),   
                Last_Purchase  = ("InvoiceDate" , "max")).reset_index()  

In [15]:
print(data["InvoiceDate"].max())

2011-12-09 12:50:00


In [16]:
reference_date = pd.Timestamp("2011-12-10")

In [17]:
customer_data["recency"] = (reference_date - customer_data["Last_Purchase"]).dt.days

In [18]:
print(customer_data[["CustomerID", "Total_Quantity" , "Total_Spending" , "Purchase_Count" ,"recency" , "Last_Purchase"]].head())

   CustomerID  Total_Quantity  Total_Spending  Purchase_Count  recency  \
0     12346.0               0            0.00               2      325   
1     12347.0            2458         4310.00             182        2   
2     12348.0            2341         1797.24              31       75   
3     12349.0             631         1757.55              73       18   
4     12350.0             197          334.40              17      310   

        Last_Purchase  
0 2011-01-18 10:17:00  
1 2011-12-07 15:52:00  
2 2011-09-25 13:13:00  
3 2011-11-21 09:51:00  
4 2011-02-02 16:01:00  


In [19]:
feature_select = customer_data[["recency" , "Total_Spending" , "Purchase_Count" ]]

In [20]:
scalling = StandardScaler()
fit_transform = scalling.fit_transform(feature_select)

In [21]:
print(fit_transform[:5])

[[ 2.31656788 -0.23041952 -0.39205597]
 [-0.88904975  0.29405454  0.39329343]
 [-0.16456032 -0.01171748 -0.26552745]
 [-0.73025754 -0.01654727 -0.08227926]
 [ 2.16770018 -0.18972715 -0.32661018]]


In [22]:
model = KMeans(n_clusters=3 , n_init=10 , random_state=1)

In [23]:
cluster = model.fit_predict(fit_transform)

In [24]:
customer_data["clusters"] = cluster

In [25]:
print(customer_data.head())

   CustomerID  Total_Quantity  Total_Spending  Purchase_Count  \
0     12346.0               0            0.00               2   
1     12347.0            2458         4310.00             182   
2     12348.0            2341         1797.24              31   
3     12349.0             631         1757.55              73   
4     12350.0             197          334.40              17   

        Last_Purchase  recency  clusters  
0 2011-01-18 10:17:00      325         1  
1 2011-12-07 15:52:00        2         0  
2 2011-09-25 13:13:00       75         0  
3 2011-11-21 09:51:00       18         0  
4 2011-02-02 16:01:00      310         1  


In [26]:
cluster_mapping = { 0 : "HIGH" ,
                    1 : "MEDIUM" ,
                    2 :  "LOW" 
}

In [27]:
customer_data["segment"] = customer_data["clusters"].map(cluster_mapping)

In [28]:
new_customer = pd.DataFrame({
    "recency": [30],
    "Total_Spending": [1500],
    "Purchase_Count": [20],
})

In [29]:
transfrom = scalling.transform(new_customer)

In [30]:
prediction = model.predict(transfrom)

In [31]:
segment = cluster_mapping[prediction[0]]

In [32]:
print("predicted clusters:", segment)

predicted clusters: HIGH


In [33]:
import pickle

pickle.dump(model, open("customer_segmentation_model.pkl", "wb"))
pickle.dump(scalling, open("customer_segmentation_scaler.pkl", "wb"))